<a href="https://colab.research.google.com/github/kilianodonell-cmd/Q3_Durban/blob/main/Field_Map_Durban.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive only when running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    print('Colab environment detected. Google Drive mounted.')
else:
    print('Local environment detected. Skipping Google Drive mount.')

print('Setup complete.')

Mounted at /content/drive
Colab environment detected. Google Drive mounted.
Setup complete.


In [2]:
import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io
import matplotlib.pyplot as plt
import pandas as pd


In [3]:
# ============================================================
# CELL 1 - Setup
# Field Map - Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
# ============================================================

# Load config (created by MCA notebook)
# Priority order:
# 1) FIELD_MAP_CONFIG_PATH environment variable
# 2) Local repo outputs
# 3) Colab default path
candidate_config_paths = []

env_config = os.environ.get('FIELD_MAP_CONFIG_PATH', '').strip()
if env_config:
    candidate_config_paths.append(env_config)

# Local workspace candidate path
candidate_config_paths.append(os.path.join(os.getcwd(), 'outputs', 'field_map_config.json'))

# Colab default candidate path
candidate_config_paths.append('/content/drive/MyDrive/Durban/outputs/field_map_config.json')

CONFIG_PATH = None
for path in candidate_config_paths:
    if os.path.exists(path):
        CONFIG_PATH = path
        break

if CONFIG_PATH is None:
    raise FileNotFoundError(
        'Could not find field_map_config.json. Set FIELD_MAP_CONFIG_PATH or generate outputs from Durban_MCA.ipynb.'
    )

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT = config['OUTPUT_ROOT']
TARGET_CRS = config['TARGET_CRS']
SCENARIOS = config['SCENARIOS']
SUIT_COLORS = config['SUIT_COLORS']
SUIT_LABELS = config['SUIT_LABELS']
AOI_FIELD = config['AOI_FIELD']
AOI_VALUE = config['AOI_VALUE']
CATCHMENTS_PATH = config['CATCHMENTS_PATH']

RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'at_risk_buildings.gpkg')
CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')

print('=' * 60)
print('FIELD MAP SETUP')
print('=' * 60)
print(f'  Config path: {CONFIG_PATH}')
print(f'  Output root: {OUTPUT_ROOT}')
print(f'  Scenarios:   {list(SCENARIOS.keys())}')

# Check that required files exist
print('\n  Checking files...')
missing = []

for scenario in SCENARIOS:
    raster_path = os.path.join(RASTERS_DIR, f'suitability_{scenario}.tif')
    if os.path.exists(raster_path):
        print(f'    ok suitability_{scenario}.tif')
    else:
        print(f'    missing suitability_{scenario}.tif')
        missing.append(raster_path)

constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if os.path.exists(constraint_path):
    print('    ok constraint_mask.tif')
else:
    print('    missing constraint_mask.tif')
    missing.append(constraint_path)

if os.path.exists(BUILDINGS_PATH):
    print('    ok at_risk_buildings.gpkg')
else:
    print('    missing at_risk_buildings.gpkg')
    missing.append(BUILDINGS_PATH)

if missing:
    print(f'\n  Warning: {len(missing)} files missing. Run Durban_MCA.ipynb first.')
else:
    print('\n  All files ready. Proceed to the map cell.')

print('=' * 60)

FIELD MAP SETUP
  Config path: /content/drive/MyDrive/Durban/outputs/field_map_config.json
  Output root: /content/drive/MyDrive/Durban/outputs
  Scenarios:   ['hazard_focused', 'balanced', 'infrastructure_focused']

  Checking files...
    ok suitability_hazard_focused.tif
    ok suitability_balanced.tif
    ok suitability_infrastructure_focused.tif
    ok constraint_mask.tif
    ok at_risk_buildings.gpkg

  All files ready. Proceed to the map cell.


In [4]:
# ============================================================
# CELL 2 - Precompute Building Factor Scores (from MCA rasters)
# ============================================================

import glob
import pandas as pd

AT_RISK_PATH = os.path.join(OUTPUT_ROOT, 'at_risk_buildings.gpkg')
FACTOR_RASTER_PATTERN = os.path.join(RASTERS_DIR, 'scored_*.tif')

if not os.path.exists(AT_RISK_PATH):
    raise FileNotFoundError(f'Missing buildings layer: {AT_RISK_PATH}')

factor_rasters = sorted(glob.glob(FACTOR_RASTER_PATTERN))
if not factor_rasters:
    raise FileNotFoundError(f'No factor rasters found matching: {FACTOR_RASTER_PATTERN}')

prepared_buildings = gpd.read_file(AT_RISK_PATH)

if 'in_constraint' not in prepared_buildings.columns:
    raise ValueError("Expected 'in_constraint' column in at_risk_buildings.gpkg")

def pick_id_column(df):
    id_candidates = [
        'building_id', 'BUILDING_ID', 'id', 'ID', 'fid', 'FID',
        'objectid', 'OBJECTID', 'osm_id', 'OSM_ID'
    ]
    for c in id_candidates:
        if c in df.columns:
            return c
    return None

popup_id_col = pick_id_column(prepared_buildings)
if popup_id_col is None:
    popup_id_col = 'building_index'
    prepared_buildings[popup_id_col] = prepared_buildings.index.astype(str)

# Sample each scored factor raster at building centroids.
for raster_path in factor_rasters:
    factor_name = os.path.splitext(os.path.basename(raster_path))[0].replace('scored_', '', 1)
    factor_col = f'factor_score_{factor_name}'

    with rasterio.open(raster_path) as src:
        src_crs = src.crs
        nodata = src.nodata

        gdf_src = prepared_buildings.to_crs(src_crs)
        centroids = gdf_src.geometry.centroid
        coords = [(geom.x, geom.y) for geom in centroids]

        sampled = [val[0] for val in src.sample(coords)]
        sampled = pd.to_numeric(pd.Series(sampled), errors='coerce')

        if nodata is not None:
            sampled = sampled.where(sampled != nodata)

        sampled = sampled.where(sampled.isin([1, 2, 3, 4, 5]))
        prepared_buildings[factor_col] = sampled.values

factor_cols = [c for c in prepared_buildings.columns if c.startswith('factor_score_')]
if not factor_cols:
    raise ValueError('No factor_score_* columns were created from scored rasters.')

def nice_factor_name(col_name):
    return col_name.replace('factor_score_', '').replace('_', ' ').strip().title()

def compute_top3_worst_text(row, cols):
    scored = []
    for c in cols:
        v = row.get(c)
        if pd.isna(v):
            continue
        fv = float(v)
        if fv < 1 or fv > 5:
            continue
        scored.append((fv, nice_factor_name(c)))
    if not scored:
        return 'Factor scores unavailable'
    scored.sort(key=lambda x: (-x[0], x[1]))
    return ', '.join([f'{name} ({int(val)})' for val, name in scored[:3]])

# Single top3_worst column — factor scores are the same regardless of scenario
prepared_buildings['top3_worst'] = prepared_buildings.apply(
    lambda row: compute_top3_worst_text(row, factor_cols),
    axis=1,
)

print('=' * 60)
print('PRECOMPUTE COMPLETE')
print('=' * 60)
print(f'Factor rasters found:   {len(factor_rasters)}')
print(f'Factor columns created: {len(factor_cols)}')
print(f'Popup ID column:        {popup_id_col}')
print('Prepared data is ready. Run next cell to create the map.')

PRECOMPUTE COMPLETE
Factor rasters found:   8
Factor columns created: 8
Popup ID column:        building_index
Prepared data is ready. Run next cell to create the map.


In [5]:
# ============================================================
# FIELD MAP - OPTIMISED
# Changes vs previous version:
#   1. Building geometry written ONCE (2 layers vs 6) — biggest size saving
#   2. Raster overlays downsampled to max 1024px
#   3. Only required columns serialised into GeoJSON
#   4. Scenario switching via JS dropdown (no layer duplication)
# ============================================================

import os
import numpy as np
import folium
from folium import plugins
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape, box
from PIL import Image
from IPython.display import IFrame, display

print('=' * 60)
print('FIELD MAP (OPTIMISED)')
print('=' * 60)

# ------------------------------------------------------------
# LOAD PREPARED DATA
# ------------------------------------------------------------
if 'prepared_buildings' not in globals():
    raise RuntimeError('Please run Cell 4 (precompute factors) before running this map cell.')
if 'popup_id_col' not in globals():
    raise RuntimeError('Missing popup_id_col. Please run Cell 4 first.')

buildings = prepared_buildings.copy()
scenario_keys = list(SCENARIOS.keys())

CONSTRAINT_RASTER = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if not os.path.exists(CONSTRAINT_RASTER):
    raise FileNotFoundError(f'Missing constraint raster: {CONSTRAINT_RASTER}')

missing_cols = [f'display_score_{s}' for s in scenario_keys if f'display_score_{s}' not in buildings.columns]
if missing_cols:
    raise ValueError('Scenario display score columns missing:\n' + '\n'.join(missing_cols))

# Reproject and simplify
buildings = buildings.to_crs(4326)
buildings['geometry'] = buildings.geometry.simplify(0.00001, preserve_topology=True)

constraint_buildings    = buildings[buildings['in_constraint'] == True].copy()
non_constraint_buildings = buildings[buildings['in_constraint'] != True].copy()

# Strip to only the columns needed for the popup — removes all factor_score_* columns
# from the serialised GeoJSON, significantly reducing file size.
display_cols = [f'display_score_{s}' for s in scenario_keys]
top3_col = 'top3_worst' if 'top3_worst' in buildings.columns else f'top3_worst_{scenario_keys[0]}'
keep_cols = [popup_id_col] + display_cols + [top3_col, 'geometry']
keep_cols = [c for c in keep_cols if c in non_constraint_buildings.columns]

non_constraint_buildings = non_constraint_buildings[keep_cols]
constraint_buildings     = constraint_buildings[[c for c in keep_cols if c in constraint_buildings.columns]]

# Load AOI boundary
catchments = gpd.read_file(CATCHMENTS_PATH)
aoi = catchments[catchments[AOI_FIELD] == AOI_VALUE].copy().to_crs(4326)
aoi['geometry'] = aoi.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# CONSTRAINT AREA FROM RASTER
# ------------------------------------------------------------
constraint_mask_gdf = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

with rasterio.open(CONSTRAINT_RASTER) as src:
    mask_data = src.read(1)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    print('Constraint raster unique values:', np.unique(mask_data))
    valid_mask = (mask_data != nodata) if nodata is not None else np.ones(mask_data.shape, dtype=bool)

    polygons = [shape(geom) for geom, value in shapes(mask_data, mask=valid_mask, transform=transform) if value == 0]
    if polygons:
        constraint_mask_gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs).to_crs(4326)
        constraint_mask_gdf['geometry'] = constraint_mask_gdf.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# MAP BASE
# ------------------------------------------------------------
center = aoi.union_all().centroid

print(f'AOI: {AOI_VALUE}')
print(f'Total buildings: {len(buildings):,}')
print(f'In-constraint: {len(constraint_buildings):,}  |  Non-constraint: {len(non_constraint_buildings):,}')
print(f'Constraint polygons: {len(constraint_mask_gdf):,}')

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=16,
    max_zoom=22,
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a> &copy; CARTO',
    control_scale=True,
)

# AOI boundary
folium.GeoJson(
    aoi,
    name='AOI boundary',
    style_function=lambda feat: {
        'fillColor': 'none', 'color': '#111111', 'weight': 2, 'dashArray': '5, 5',
    },
).add_to(m)

# ------------------------------------------------------------
# RASTER OVERLAYS — downsampled to max 1024px
# ------------------------------------------------------------
MAX_RASTER_PX = 1024

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

for idx, scenario in enumerate(scenario_keys):
    scenario_raster = os.path.join(RASTERS_DIR, f'suitability_{scenario}.tif')
    if not os.path.exists(scenario_raster):
        print(f'  Skipping scenario mask (missing): {scenario}')
        continue

    with rasterio.open(scenario_raster) as src:
        arr = src.read(1)
        nodata = src.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, 0, arr)

        rgba = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)
        for class_value in [1, 2, 3, 4, 5]:
            class_mask = arr == class_value
            if class_mask.any():
                r, g, b = hex_to_rgb(SUIT_COLORS[class_value - 1])
                rgba[class_mask, 0] = r
                rgba[class_mask, 1] = g
                rgba[class_mask, 2] = b
                rgba[class_mask, 3] = 125

        # Downsample large rasters before embedding
        img = Image.fromarray(rgba, mode='RGBA')
        orig_size = img.size
        if max(img.size) > MAX_RASTER_PX:
            img.thumbnail((MAX_RASTER_PX, MAX_RASTER_PX), Image.LANCZOS)
        print(f'  Raster {scenario}: {orig_size} → {img.size}')
        rgba_out = np.array(img)

        raster_bounds_geom = gpd.GeoSeries(
            [box(src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)],
            crs=src.crs,
        ).to_crs(4326)
        minx, miny, maxx, maxy = raster_bounds_geom.total_bounds

        folium.raster_layers.ImageOverlay(
            image=rgba_out,
            bounds=[[miny, minx], [maxy, maxx]],
            name=f'Scenario mask: {scenario}',
            opacity=0.65,
            interactive=False,
            show=(idx == 0),
            cross_origin=False,
            zindex=1,
        ).add_to(m)

# Constraint area polygon layer
if len(constraint_mask_gdf) > 0:
    folium.GeoJson(
        constraint_mask_gdf,
        name='Filter: Constraint area',
        style_function=lambda feat: {
            'fillColor': '#dc2626', 'color': '#991b1b', 'weight': 0.7, 'fillOpacity': 0.28,
        },
        tooltip='Constraint area',
        show=True,
    ).add_to(m)

# ------------------------------------------------------------
# BUILDING LAYERS — 2 layers instead of 6
# Popup shows: building ID, all scenario scores, top 3 worst factors
# Non-constraint layer is re-styled via JS when scenario changes
# ------------------------------------------------------------
popup_fields  = [popup_id_col] + display_cols + [top3_col]
popup_aliases = (
    ['Building ID:']
    + [f'{s.replace("_", " ").title()} score:' for s in scenario_keys]
    + ['Top 3 worst factors:']
)

first_scenario = scenario_keys[0]

# Initial Python style (first scenario) — JS takes over after scenario change
def nc_initial_style(feature, s=first_scenario):
    score = feature['properties'].get(f'display_score_{s}')
    if score in [1, 2, 3, 4, 5]:
        return {'fillColor': SUIT_COLORS[int(score) - 1], 'color': '#2d2d2d', 'weight': 0.2, 'fillOpacity': 0.85}
    return {'fillColor': '#9e9e9e', 'color': '#2d2d2d', 'weight': 0.2, 'fillOpacity': 0.85}

nc_layer = folium.GeoJson(
    non_constraint_buildings,
    name='Buildings (non-constraint)',
    style_function=nc_initial_style,
    popup=folium.GeoJsonPopup(
        fields=popup_fields,
        aliases=popup_aliases,
        localize=True,
        labels=True,
    ),
    tooltip=folium.GeoJsonTooltip(
        fields=[f'display_score_{first_scenario}'],
        aliases=['Class:'],
        sticky=False,
    ),
    show=True,
)
nc_layer.add_to(m)
nc_layer_id = nc_layer.get_name()

# Constraint buildings — static dark red style, same popup
if len(constraint_buildings) > 0:
    folium.GeoJson(
        constraint_buildings,
        name='Filter: Buildings in constraint',
        style_function=lambda feat: {
            'fillColor': '#7f1a1a', 'color': '#3f0a0a', 'weight': 0.25, 'fillOpacity': 0.9,
        },
        popup=folium.GeoJsonPopup(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            labels=True,
        ),
        show=True,
    ).add_to(m)

# ------------------------------------------------------------
# MAP TOOLS
# ------------------------------------------------------------
plugins.Fullscreen().add_to(m)
plugins.MeasureControl(position='topleft', primary_length_unit='meters').add_to(m)
plugins.MousePosition(position='topright').add_to(m)
plugins.Draw(
    export=True,
    filename='candidate_area.geojson',
    position='topleft',
    draw_options={'polyline': False, 'rectangle': True, 'circle': False, 'marker': False, 'circlemarker': False},
    edit_options={'edit': True, 'remove': True},
).add_to(m)

# ------------------------------------------------------------
# SCENARIO SELECTOR + JS re-style
# ------------------------------------------------------------
suit_colors_js = '{' + ', '.join([f'"{i+1}": "{c}"' for i, c in enumerate(SUIT_COLORS)]) + '}'

scenario_options_html = ''.join([
    f'<option value="{s}"{"" if i > 0 else " selected"}>{s.replace("_", " ").title()}</option>'
    for i, s in enumerate(scenario_keys)
])

selector_html = f"""
<div id="scenario-selector" style="
    position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
    z-index: 9999; background: white; border: 1px solid #d1d5db;
    border-radius: 6px; padding: 6px 14px; font-size: 13px;
    font-family: sans-serif; box-shadow: 0 2px 6px rgba(0,0,0,0.15);">
  <label style="font-weight:600; margin-right:8px;">Scenario:</label>
  <select id="scenario-select" onchange="updateBuildingScenario(this.value)"
          style="font-size:13px; padding:2px 6px; border-radius:4px;">
    {scenario_options_html}
  </select>
</div>

<script>
  var _suitColors = {suit_colors_js};
  window.activeScenario = '{first_scenario}';

  function updateBuildingScenario(scenario) {{
    window.activeScenario = scenario;
    var lyr = window['{nc_layer_id}'];
    if (lyr) {{
      lyr.setStyle(function(feature) {{
        var score = String(feature.properties['display_score_' + scenario]);
        return {{
          fillColor: _suitColors[score] || '#9e9e9e',
          color: '#2d2d2d',
          weight: 0.2,
          fillOpacity: 0.85
        }};
      }});
    }}
  }}
</script>
"""
m.get_root().html.add_child(folium.Element(selector_html))

# ------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------
suit_swatches = ''.join([
    f'<div><span style="display:inline-block;width:16px;height:10px;background:{SUIT_COLORS[i]};margin-right:5px;vertical-align:middle;"></span>{SUIT_LABELS[i]}</div>'
    for i in range(5)
])

legend_html = f"""
<div style="position: fixed; bottom: 18px; left: 18px; z-index: 9999; background: white;
     border: 1px solid #d1d5db; border-radius: 6px; padding: 10px 12px; font-size: 12px; max-width: 300px;">
  <div style="font-weight:700; margin-bottom:5px;">Housing Suitability POC</div>
  <div style="margin-bottom:3px;">Use the <b>Scenario</b> dropdown (top) to switch building colours.</div>
  <div style="margin-bottom:3px;">Toggle <b>Scenario mask</b> layers for raster surfaces.</div>
  <div style="margin-bottom:6px;">Click a building for ID, all scenario scores, top 3 worst factors.</div>
  <div style="font-weight:600; margin-bottom:3px;">Suitability Classes</div>
  {suit_swatches}
  <hr style="margin:6px 0;">
  <div><span style="display:inline-block;width:16px;height:10px;background:#dc2626;margin-right:5px;vertical-align:middle;"></span>Constraint area</div>
  <div><span style="display:inline-block;width:16px;height:10px;background:#7f1a1a;margin-right:5px;vertical-align:middle;"></span>Buildings in constraint</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
output_dir = os.path.join(OUTPUT_ROOT, 'outputs')
os.makedirs(output_dir, exist_ok=True)
output_map = os.path.join(output_dir, 'field_map_durban.html')
m.save(output_map)

size_mb = os.path.getsize(output_map) / (1024 * 1024)
print(f'\nMap saved: {output_map}')
print(f'File size: {size_mb:.1f} MB')

display(IFrame(output_map, width='100%', height='680'))

FIELD MAP (OPTIMISED)


Constraint raster unique values: [0 1]
AOI: Mzinyati Stream
Total buildings: 22,196
In-constraint: 4,231  |  Non-constraint: 17,965
Constraint polygons: 124


/tmp/ipykernel_2520/136755858.py:145: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(rgba, mode='RGBA')


  Raster hazard_focused: (348, 402) → (348, 402)


/tmp/ipykernel_2520/136755858.py:145: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(rgba, mode='RGBA')


  Raster balanced: (348, 402) → (348, 402)


/tmp/ipykernel_2520/136755858.py:145: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(rgba, mode='RGBA')


  Raster infrastructure_focused: (348, 402) → (348, 402)

Map saved: /content/drive/MyDrive/Durban/outputs/outputs/field_map_durban.html
File size: 13.4 MB
